# From a retained Carvana page to a vehicle record

**Question:** where do a VIN, listing ID and asking price in our table come from?
Run All reads local files and changes no evidence. Start with the real retained
three-record browser sample, then use the optional invented CSV to learn validation.

## Settings

The next cell names both input files. Keep the retained fixture selected for this
walkthrough; the synthetic file is a separate learning input. Neither is a full
inventory snapshot. Prices are asking prices in USD and observation times are UTC.

**Learning order:** **00 -> 10 -> 11 -> 20 -> 24 -> 30**. This first notebook is a
historical source-reading lesson, not a current inventory review. The real fixture
contains three browser records observed on September 8, 2026 at 01:44:11.591 UTC;
all rows belong to that selected capture, with no current-date substitution.

| What to change / inspect | What it means |
| --- | --- |
| `retained_path` in the next cell | The saved real browser sample; its exact local path is printed below. |
| `normalized_retained` | The parser's VIN, listing and asking-price rows from that one capture. |
| `source_file`, in the optional example | Invented CSV rows for practicing validation; not Carvana observations. |
| Empty duplicate table / missing values | No duplicate key in this fixture / an unknown source field. Neither establishes a sale. |

Changing a path only selects a local file. An absent file is an input problem to
inspect; this lesson will not fetch a replacement.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if (ROOT / "vehicle" / "src").is_dir():
    ROOT = ROOT / "vehicle"
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src" / "vehicle_tracker").is_dir(), "Launch from researchOS, vehicle/, or vehicle/notebooks/."
sys.path.insert(0, str(ROOT / "src"))
import vehicle_tracker

# Read-only inputs. The second file is used only in the optional example.
retained_path = ROOT / 'tests/fixtures/carvana_browser_sample_20260907.json'
source_file = ROOT / 'tests/fixtures/synthetic_listings.csv'


## 1. Inspect the source fields, then their normalized names

Follow listing **4474057**, VIN **3FA6P0D94ER264351**, in both tables below.
The retained record has `offers.price = 14590`; the parser produces
`asking_price_usd = 14590`. It also maps `vehicleIdentificationNumber` to `vin`
and extracts the listing ID from `offers.url`.

This three-record sample was observed at **2026-09-08 01:44:11.591 UTC**.
The source's `InStock` value is retained as `availability_native`; it does not
establish a completed transaction. Continue to [Notebook 10](10_carvana_inventory.ipynb)
for a complete declared query and its coverage checks. The [audit map](../docs/audit_guide.md)
locates the readers and parsers.


In [ ]:
import json
from vehicle_tracker.carvana import parse_capture
retained = json.loads(retained_path.read_text(encoding='utf-8'))
display(pd.json_normalize(retained['records']).head())
normalized_retained = parse_capture(retained)
display(normalized_retained[['listing_id', 'vin', 'asking_price_usd', 'availability_native', 'source_url']])
print('Retained observation:', retained['captured_at_utc'], retained_path)
print('Sample only; no population coverage or sale is established.')

### Try one real-row calculation

Locate VIN `3FA6P0D94ER264351` in both displayed tables. In a new scratch cell, use
`normalized_retained.loc[normalized_retained.vin.eq('3FA6P0D94ER264351'), 'asking_price_usd']`.
You should recover **14590**, matching `offers.price`. Dividing by 1,000 gives
**14.59 thousand USD**; this changes the unit, not the underlying price or evidence.
Next, choose another retained VIN and repeat the lookup. No request or save is needed.


## Optional learning example: validate an invented CSV

The following rows are synthetic; their statuses are not a verified retailer
taxonomy. The observation key is **retailer + listing ID + capture time**.
The same VIN at different retailers remains separate evidence. Inspect missing
fields and duplicate keys before reading the normalized table.


In [ ]:
observed = pd.read_csv(source_file)
print('SYNTHETIC EXAMPLE ONLY:', source_file)
display(observed)

key = ["retailer", "listing_id", "observed_at_utc"]
duplicate_rows = observed[observed.duplicated(key, keep=False)]
missing_keys = observed[key].isna().any(axis=1)
display(observed.isna().sum().rename("missing_values").to_frame())
display(duplicate_rows)
assert not missing_keys.any(), "Missing observation key: inspect the source."
assert duplicate_rows.empty, "Duplicate observation keys: inspect before analysis."

snapshots = observed.copy()
snapshots["observed_at_utc"] = pd.to_datetime(snapshots["observed_at_utc"], utc=True, errors="raise")
snapshots["asking_price_usd"] = pd.to_numeric(snapshots["asking_price_usd"], errors="raise")
# Parse dates before the second check: equivalent timestamp spellings are one key.
assert not snapshots.duplicated(key).any(), "Duplicate normalized observation keys."
display(snapshots)


### Count the latest synthetic observations

Filter to the latest capture, then group by retailer and native status. A missing
price stays missing; a zero count means no row in this fixture's group. These
counts describe the invented example only. Real absence comparisons additionally
require complete, comparable collections, as shown in [Notebook 20](20_carvana_history_analysis.ipynb).


In [ ]:
latest_capture = snapshots["observed_at_utc"].max()
latest = snapshots[snapshots["observed_at_utc"].eq(latest_capture)].copy()
listing_counts = latest.groupby(["retailer", "native_status"], dropna=False).size().rename("observed_listings").reset_index()
display(latest)
display(listing_counts)
print("Synthetic listing counts only; no sale is inferred from this fixture.")
